In [1]:
import pandas as pd
import os
import numpy as np
import json

In [2]:
Dir_dataset = '/work/desai-lab/ABC_dataset'
Dir_github = '/work/xy6/Project/ABC/github'
repo = 'ABC_valence'
pipeline = 'passage_glm_sm6AVmot6_gpt'
Dir_deriv = os.path.join(Dir_github,'derivatives')
Dir_working = os.path.join(Dir_github,repo,'scripts','fMRI','passage',pipeline,)
rater = 'gpt-4o-2024-08-06'
modal = 'text'
model_rating = f"{modal}_{rater}"
Dir_timing = os.path.join(Dir_deriv,'behavioral','timings',rater)


In [3]:
dfs_eprime_raw = pd.read_csv(os.path.join(Dir_dataset,'codes','fmri_passage_1001-1362.csv'))
dfs_eprime = dfs_eprime_raw.loc[~dfs_eprime_raw['passageReal.OnsetTime'].isna()].copy()
dfs_eprime['delay'] = dfs_eprime['passageReal.OnsetTime']-dfs_eprime['readyFix1.RTTime']
dfs_eprime['delay'] = dfs_eprime['delay'] /1000
dfs_eprime = dfs_eprime[['Subject','Session','name','delay']]


# select participants for analysis
df_master = pd.read_csv(os.path.join(Dir_dataset,'participants_master.csv'))
df_subj = df_master.copy()
df_subj = df_subj.loc[~df_subj['Passage_eprime'].isna()].loc[~df_subj['Passage_fmri'].isna()]
df_subj = df_subj.loc[df_subj['visit']==1]
df_subj['story'] = df_subj['story'].str.replace('.wav','')
df_subj['story'] = df_subj['story'].str.replace('BradfordBull','BradfordBoar')
df_subj['story'] = df_subj['story'].str.replace('Roswell','Rosewell')
df_subj['story'] = df_subj['story'].str.replace('BulliesPronouns','BulliesPronoun')
df_subj.loc[df_subj['story'].str.endswith('Bradford'),'story'] = df_subj.loc[df_subj['story'].str.endswith('Bradford'),'story'].str.replace('Bradford','BradfordBoar')
df_subj['audio'] = df_subj['story']

df_subj[['voice','story','story_postfix']] = df_subj['story'].str.split('_',expand=True)
# df_subj.loc[df_subj['story'].str.contains('EmotionSentences'),'story']='EmotionSentences_953'
df_subj = df_subj[['subid','visit','voice','story','audio','Passage_fmri','T1','age_dayofday1testing','sex','handedness','first_language']]
df_subj.rename({'age_dayofday1testing':'age'},axis=1,inplace=True)

df_subj = pd.merge(left=df_subj,right=dfs_eprime,left_on=['subid','visit'],right_on=['Subject','Session'],how='left')
df_subj

,subid,visit,voice,story,audio,Passage_fmri,T1,age,sex,handedness,first_language,Subject,Session,name,delay
0,1006,1.0,Brandi,TaleofTwoCountries,Brandi_TaleofTwoCountries,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,61.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN
1,1007,1.0,Jack,ProfessionalDecline,Jack_ProfessionalDecline,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,69.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN
2,1008,1.0,Jacob,FairyTree,Jacob_FairyTree,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,77.0,2.0,2.0,1.0,1008.0,1.0,Jacob_FairyTree.wav,58.207
3,1009,1.0,Jacob,HowCouldYou,Jacob_HowCouldYou,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,75.0,2.0,2.0,1.0,1009.0,1.0,Jacob_HowCouldYou.wav,58.201
4,1012,1.0,Jacob,UserFriendly,Jacob_UserFriendly,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,62.0,2.0,2.0,1.0,1012.0,1.0,Jacob_UserFriendly.wav,58.211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,1318,1.0,Allison,MetaphorsRepeatedNames,Allison_MetaphorsRepeatedNames,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,71.0,2.0,2.0,1.0,1318.0,1.0,Allison_MetaphorsRepeatedNames.wav,58.199
168,1320,1.0,Allison,BulliesRepeatedNames,Allison_BulliesRepeatedNames,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,52.0,2.0,2.0,1.0,1320.0,1.0,Allison_BulliesRepeatedNames.wav,58.207
169,1321,1.0,Jacob,Aqua,Jacob_Aqua,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,34.0,2.0,NaN,NaN,1321.0,1.0,Jacob_Aqua.wav,58.206
170,1322,1.0,Brandi,BradfordBoar,Brandi_BradfordBoar,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,53.0,2.0,2.0,1.0,1322.0,1.0,Brandi_BradfordBull.wav,58.211


In [4]:
def extract_compcor(path_confounds,tissue,N_comps):
    df_confounds = pd.read_csv(path_confounds,sep='\t')
    path_confounds_json = path_confounds.replace('.tsv','.json')
    f = open (path_confounds_json, "r")
    data = json.loads(f.read())
    f.close()
    
    List_compcor_all = [x for x in df_confounds.columns if 'comp_cor' in x and 't_comp_cor' not in x]
    List_aCompCor = []
    df = []
    List_cosine = [x for x in df_confounds.columns if 'cosine' in x]
    if len(List_cosine)>0 and len(List_compcor_all)>0:


        for x in List_compcor_all:
            if data[x]['Mask'] == tissue and data[x]['Method']=='aCompCor':
                List_aCompCor.append(x)
                df.append(pd.DataFrame({'c':x,'Mask':data[x]['Mask'],'Method':data[x]['Method'],
                                      'Retained':data[x]['Retained'],'SingularValue':data[x]['SingularValue'],
                                      'VarExp':data[x]['VarianceExplained']},index=[0]))
        df = pd.concat(df,ignore_index=True)
        df = df.loc[df.Retained==1].sort_values(['SingularValue'],ascending=False)

        if len(List_aCompCor)>=N_comps:
            List_aCompCor = List_aCompCor[:N_comps]


        List_cosine = [x for x in df_confounds.columns if 'cosine' in x]
        
    else:
        print('Failed to extract cosine or aCompCor. Check {}'.format(path_confounds))
#         print('Failed to extract cosine or aCompCor.\n')
        
    return(df,List_cosine, List_aCompCor)

In [14]:
# task = 'Passage'

FWHM = 6
N_CSF = 0
N_WM = 0
model_confounds = f'mot{6}CSF{N_CSF}WM{N_WM}'
model_glm = f'sm{FWHM}_AV{model_confounds}'
task = 'Passage'
Dir_results = os.path.join(Dir_github,'derivatives','fmri',pipeline,model_rating,'firstlevel',model_glm)

for i,row in df_subj.iterrows():
    subid = int(row['subid'])
    visit = int(row['visit'])
    story = row['story']
    # smoothed data

    Dir_results_subj = os.path.join(Dir_results,f"sub-{subid}")
    if not os.path.exists(Dir_results_subj):
        os.makedirs(Dir_results_subj)
    # if subid <= 1197:
    #     postfix = 'sm6_clean_6motCSF5WM5polort4sm6'
    #     path = os.path.join(Dir_dataset,'derivatives','afni-smooth',f'sub-{subid}',f'ses-visit0{visit}','func',
    #                         f"sub-{subid}_ses-visit0{visit}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-{postfix}.nii.gz")
    # else:
    postfix = f'sm{FWHM}'
    path = os.path.join(Dir_github,'derivatives','fmri','afni_smooth-bold',f'sub-{subid}',
                        f"sub-{subid}_ses-visit0{visit}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-{postfix}_bold.nii.gz")
    if os.path.exists(path):
        df_subj.loc[i,f'{task}_smooth'] = path
    else:
        df_subj.loc[i,f'{task}_smooth'] = np.nan

    # mask 
    path = os.path.join(Dir_dataset,'derivatives','fmriprep',f'sub-{subid}',f'ses-visit0{visit}','func',
                       f"sub-{subid}_ses-visit0{visit}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz")
    if os.path.exists(path):
        df_subj.loc[i,f'{task}_mask'] = path
    else:
        df_subj.loc[i,f'{task}_mask'] = np.nan

    # timing
    path = os.path.join(Dir_timing,f"timing_{rater}_{story}_bySeg.csv")
    if os.path.exists(path):
        df_subj.loc[i,f'{task}_timing'] = path
    else:
        df_subj.loc[i,f'{task}_timing'] = np.nan

    # confounding
    path = os.path.join(Dir_dataset,'derivatives','fmriprep',f'sub-{subid}',f'ses-visit0{visit}','func',
                        f"sub-{subid}_ses-visit0{visit}_task-{task}_desc-confounds_timeseries.tsv")

    path_output = os.path.join(Dir_results,f"sub-{subid}",
                              f"sub-{subid}_ses-visit0{visit}_task-{task}_confounds.txt")
    if os.path.exists(path_output):
        df_subj.loc[i,f'{task}_confounds'] = path_output
        continue
        
    df_confounds = pd.read_csv(path,sep='\t')
    df_tmp,List_cosine,List_aCompCor_CSF = extract_compcor(path,'CSF',N_CSF)
    df_tmp,List_cosine,List_aCompCor_WM = extract_compcor(path,'WM',N_WM)
    
    List_motion = ['trans_x', 'trans_y', 'trans_z','rot_x', 'rot_y', 'rot_z']
    if (N_CSF==0) & (N_WM==0):
        df_confounds_model = df_confounds[List_motion]
        df_subj.loc[i,f'{task}_confounds'] = path_output
        df_confounds_model.to_csv(path_output,sep='\t',header=None,index=False)
        df_confounds_model.to_csv(path_output.replace('.txt','.csv'),index=False)
    else:
        df_confounds_model = df_confounds[List_motion+List_cosine+List_aCompCor_CSF+List_aCompCor_WM]
        if len(List_aCompCor_CSF) ==0 or len(List_aCompCor_WM)==0:
            df_subj.loc[i,f'{task}_confounds'] = np.nan
        else:
            df_subj.loc[i,f'{task}_confounds'] = path_output
            df_confounds_model.to_csv(path_output,sep='\t',header=None,index=False)
            df_confounds_model.to_csv(path_output.replace('.txt','.csv'),index=False)

df_output = df_subj.loc[~df_subj['Passage_smooth'].isna()].loc[~df_subj['Passage_confounds'].isna()].loc[~df_subj['delay'].isna()]

# df_output = df_output[['subid','visit','voice','story','audio','delay']+[x for x in df_output.columns if task in x]].dropna()
df_output.to_csv(os.path.join(Dir_working,f'df_subjects_N{df_output.shape[0]}.csv'),index=False)
df_output

Failed to extract cosine or aCompCor. Check /work/desai-lab/ABC_dataset/derivatives/fmriprep/sub-1147/ses-visit01/func/sub-1147_ses-visit01_task-Passage_desc-confounds_timeseries.tsv
Failed to extract cosine or aCompCor. Check /work/desai-lab/ABC_dataset/derivatives/fmriprep/sub-1147/ses-visit01/func/sub-1147_ses-visit01_task-Passage_desc-confounds_timeseries.tsv
Failed to extract cosine or aCompCor. Check /work/desai-lab/ABC_dataset/derivatives/fmriprep/sub-1226/ses-visit01/func/sub-1226_ses-visit01_task-Passage_desc-confounds_timeseries.tsv
Failed to extract cosine or aCompCor. Check /work/desai-lab/ABC_dataset/derivatives/fmriprep/sub-1226/ses-visit01/func/sub-1226_ses-visit01_task-Passage_desc-confounds_timeseries.tsv


,subid,visit,voice,story,audio,Passage_fmri,T1,age,sex,handedness,first_language,Subject,Session,name,delay,Passage_smooth,Passage_mask,Passage_timing,Passage_confounds
2,1008,1.0,Jacob,FairyTree,Jacob_FairyTree,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,77.0,2.0,2.0,1.0,1008.0,1.0,Jacob_FairyTree.wav,58.207,/work/xy6/Project/ABC/github/derivatives/fmri/...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/xy6/Project/ABC/github/derivatives/behav...,/work/xy6/Project/ABC/github/derivatives/fmri/...
3,1009,1.0,Jacob,HowCouldYou,Jacob_HowCouldYou,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,75.0,2.0,2.0,1.0,1009.0,1.0,Jacob_HowCouldYou.wav,58.201,/work/xy6/Project/ABC/github/derivatives/fmri/...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/xy6/Project/ABC/github/derivatives/behav...,/work/xy6/Project/ABC/github/derivatives/fmri/...
4,1012,1.0,Jacob,UserFriendly,Jacob_UserFriendly,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,62.0,2.0,2.0,1.0,1012.0,1.0,Jacob_UserFriendly.wav,58.211,/work/xy6/Project/ABC/github/derivatives/fmri/...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/xy6/Project/ABC/github/derivatives/behav...,/work/xy6/Project/ABC/github/derivatives/fmri/...
5,1014,1.0,Brandi,BradfordBoar,Brandi_BradfordBoar,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,73.0,1.0,2.0,1.0,1014.0,1.0,Brandi_BradfordBull.wav,58.197,/work/xy6/Project/ABC/github/derivatives/fmri/...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/xy6/Project/ABC/github/derivatives/behav...,/work/xy6/Project/ABC/github/derivatives/fmri/...
6,1016,1.0,Jacob,EmpireState,Jacob_EmpireState,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,65.0,2.0,2.0,1.0,1016.0,1.0,Jacob_EmpireState.wav,58.204,/work/xy6/Project/ABC/github/derivatives/fmri/...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/xy6/Project/ABC/github/derivatives/behav...,/work/xy6/Project/ABC/github/derivatives/fmri/...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,1318,1.0,Allison,MetaphorsRepeatedNames,Allison_MetaphorsRepeatedNames,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,71.0,2.0,2.0,1.0,1318.0,1.0,Allison_MetaphorsRepeatedNames.wav,58.199,/work/xy6/Project/ABC/github/derivatives/fmri/...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/xy6/Project/ABC/github/derivatives/behav...,/work/xy6/Project/ABC/github/derivatives/fmri/...
168,1320,1.0,Allison,BulliesRepeatedNames,Allison_BulliesRepeatedNames,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,52.0,2.0,2.0,1.0,1320.0,1.0,Allison_BulliesRepeatedNames.wav,58.207,/work/xy6/Project/ABC/github/derivatives/fmri/...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/xy6/Project/ABC/github/derivatives/behav...,/work/xy6/Project/ABC/github/derivatives/fmri/...
169,1321,1.0,Jacob,Aqua,Jacob_Aqua,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,34.0,2.0,NaN,NaN,1321.0,1.0,Jacob_Aqua.wav,58.206,/work/xy6/Project/ABC/github/derivatives/fmri/...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/xy6/Project/ABC/github/derivatives/behav...,/work/xy6/Project/ABC/github/derivatives/fmri/...
170,1322,1.0,Brandi,BradfordBoar,Brandi_BradfordBoar,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,53.0,2.0,2.0,1.0,1322.0,1.0,Brandi_BradfordBull.wav,58.211,/work/xy6/Project/ABC/github/derivatives/fmri/...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/xy6/Project/ABC/github/derivatives/behav...,/work/xy6/Project/ABC/github/derivatives/fmri/...
